# MIMESIS — Entrenamiento de `PoseMLP`, Ablación PDI y Comparativa de Modelos

Notebook oficial de entrenamiento y evaluación cuantitativa para el proyecto final de **Procesamiento Digital de Imágenes (UAO · 2026-2)**.

### Qué hace este notebook
1. Lee el dataset real del equipo en Google Drive (`belalcazar`, `discobolo`, `shakespeare`, respetando la separación `train / val / test` sin fuga de datos).
2. **Experimento de Ablación PDI:** compara la tasa de detección y visibilidad promedio de MediaPipe BlazePose **sin preprocesamiento vs. con CLAHE (LAB) + Letterboxing**.
3. Extrae los **12 ángulos corporales invariantes** usando exactamente la misma lógica de producción (`backend/pose.py`, incluido el fallback `_compute_head_tilt` en perfil).
4. Entrena y evalúa **3 modelos** sobre los mismos datos:
   - `RandomForestClassifier` (ML clásico)
   - `SVC (RBF)` (ML clásico)
   - `PoseMLP` (PyTorch — arquitectura exacta de `backend/evaluator.py`)
5. Genera la **tabla comparativa** frente al *baseline* CNN RGB del Parcial 2 (83.3% val), **matriz de confusión**, **Macro-F1** y exporta `pose_mlp.pt` listo para copiar en `backend/models/pose_mlp.pt`.

In [ ]:
!pip install -q mediapipe>=0.10.30 opencv-python-headless scikit-learn matplotlib seaborn torch

In [ ]:
import math
import urllib.request
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.svm import SVC

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE, "| MediaPipe:", mp.__version__)

## 1. Montar Google Drive y configurar ruta del dataset
Ajusta `DATASET_ROOT` a la carpeta donde tienes las subcarpetas `train/`, `val/`, `test/` (o directamente las 3 clases si aún no están divididas; el script soporta ambos formatos).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajusta esta ruta a tu carpeta en Drive (ej. '/content/drive/MyDrive/PDI - Parcial2/dataset')
DATASET_ROOT = Path("/content/drive/MyDrive/PDI - Parcial2")

SCULPTURE_CLASSES = ["belalcazar", "discobolo", "shakespeare"]
# Alias aceptados para los nombres de carpeta en Drive (insensible a mayúsculas/tildes)
CLASS_ALIASES = {
    "belalcazar": ["belalcazar", "belalcázar", "sebastian", "sebastian_de_belalcazar"],
    "discobolo": ["discobolo", "discóbolo", "el_discobolo"],
    "shakespeare": ["shakespeare", "william_shakespeare"],
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

print("DATASET_ROOT existe:", DATASET_ROOT.exists())
if DATASET_ROOT.exists():
    print("Contenido:", [p.name for p in DATASET_ROOT.iterdir()])

## 2. Pipeline PDI y Extracción de 12 Ángulos (idéntico a `backend/pdi.py` y `backend/pose.py`)

In [ ]:
FEATURE_ORDER = [
    "head_tilt", "neck_angle",
    "torso_lean", "shoulder_level",
    "left_elbow", "right_elbow", "left_shoulder", "right_shoulder",
    "left_hip", "right_hip", "left_knee", "right_knee",
]

def apply_adaptive_clahe(img_bgr: np.ndarray, dark_threshold: float = 115.0, low_contrast_std: float = 42.0):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    if float(np.mean(l_ch)) < dark_threshold or float(np.std(l_ch)) < low_contrast_std:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        merged = cv2.merge((clahe.apply(l_ch), a_ch, b_ch))
        return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), True
    return img_bgr, False

def letterbox_resize(img_bgr: np.ndarray, target_size: int = 640):
    h, w = img_bgr.shape[:2]
    scale = min(target_size / w, target_size / h)
    nw, nh = int(round(w * scale)), int(round(h * scale))
    resized = cv2.resize(img_bgr, (nw, nh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
    canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    px, py = (target_size - nw) // 2, (target_size - nh) // 2
    canvas[py:py+nh, px:px+nw] = resized
    return canvas, (w, h, target_size, scale, px, py)

def unletterbox_point(x_norm: float, y_norm: float, meta):
    orig_w, orig_h, target_size, scale, pad_x, pad_y = meta
    orig_x = (x_norm * target_size - pad_x) / (scale * orig_w)
    orig_y = (y_norm * target_size - pad_y) / (scale * orig_h)
    return float(np.clip(orig_x, 0.0, 1.0)), float(np.clip(orig_y, 0.0, 1.0))

def _angle_3pts(a, b, c) -> float:
    ba, bc = a - b, c - b
    nba, nbc = np.linalg.norm(ba), np.linalg.norm(bc)
    if nba < 1e-6 or nbc < 1e-6:
        return 0.0
    return float(math.degrees(math.acos(np.clip(np.dot(ba, bc) / (nba * nbc), -1.0, 1.0))))

def _line_angle_vertical(p_top, p_bottom) -> float:
    vec = p_top - p_bottom
    norm = np.linalg.norm(vec)
    if norm < 1e-6:
        return 0.0
    return float(math.degrees(math.acos(np.clip(np.dot(vec, np.array([0.0, -1.0])) / norm, -1.0, 1.0))))

def _line_angle_horizontal(p_left, p_right) -> float:
    vec = p_right - p_left
    return float(abs(math.degrees(math.atan2(vec[1], abs(vec[0]) + 1e-6))))

def _compute_head_tilt(landmarks, pts) -> float:
    vis_l = float(landmarks[7].get("visibility", 1.0))
    vis_r = float(landmarks[8].get("visibility", 1.0))
    shoulder_w = float(np.linalg.norm(pts[11] - pts[12]))
    ear_dx = float(abs(pts[7][0] - pts[8][0]))
    if vis_l >= 0.45 and vis_r >= 0.45 and (shoulder_w < 1e-4 or ear_dx >= 0.20 * shoulder_w):
        return _line_angle_horizontal(pts[7], pts[8])
    best_ear = 7 if vis_l >= vis_r else 8
    return _line_angle_horizontal(pts[best_ear], pts[0])

def extract_feature_vector(landmarks) -> np.ndarray:
    pts = np.array([[lm["x"], lm["y"]] for lm in landmarks], dtype=np.float32)
    mid_sh = (pts[11] + pts[12]) / 2.0
    mid_hip = (pts[23] + pts[24]) / 2.0
    angles = {
        "head_tilt": _compute_head_tilt(landmarks, pts),
        "neck_angle": _line_angle_vertical(pts[0], mid_sh),
        "torso_lean": _line_angle_vertical(mid_sh, mid_hip),
        "shoulder_level": _line_angle_horizontal(pts[11], pts[12]),
        "left_elbow": _angle_3pts(pts[11], pts[13], pts[15]),
        "right_elbow": _angle_3pts(pts[12], pts[14], pts[16]),
        "left_shoulder": _angle_3pts(pts[23], pts[11], pts[13]),
        "right_shoulder": _angle_3pts(pts[24], pts[12], pts[14]),
        "left_hip": _angle_3pts(pts[11], pts[23], pts[25]),
        "right_hip": _angle_3pts(pts[12], pts[24], pts[26]),
        "left_knee": _angle_3pts(pts[23], pts[25], pts[27]),
        "right_knee": _angle_3pts(pts[24], pts[26], pts[28]),
    }
    return np.array([angles[k] / 180.0 for k in FEATURE_ORDER], dtype=np.float32)

def mirror_feature_vector(vec: np.ndarray) -> np.ndarray:
    """Intercambia izquierdo/derecho (índices 4<->5, 6<->7, 8<->9, 10<->11) para data augmentation en train."""
    m = vec.copy()
    for i, j in ((4, 5), (6, 7), (8, 9), (10, 11)):
        m[i], m[j] = vec[j], vec[i]
    return m

In [ ]:
# Descargar PoseLandmarker (Tasks API)
TASK_MODEL_PATH = Path("/content/pose_landmarker_full.task")
if not TASK_MODEL_PATH.exists():
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"
    urllib.request.urlretrieve(url, TASK_MODEL_PATH)

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

landmarker = mp_vision.PoseLandmarker.create_from_options(
    mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_buffer=TASK_MODEL_PATH.read_bytes()),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.30,
    )
)

def detect_pose(img_bgr: np.ndarray, use_pdi: bool = True):
    if use_pdi:
        enh, _ = apply_adaptive_clahe(img_bgr)
        boxed, meta = letterbox_resize(enh, 640)
        rgb = cv2.cvtColor(boxed, cv2.COLOR_BGR2RGB)
    else:
        meta = None
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
    if not res.pose_landmarks:
        return None
    lms = []
    vis_list = []
    for lm in res.pose_landmarks[0]:
        ox, oy = unletterbox_point(lm.x, lm.y, meta) if meta else (float(lm.x), float(lm.y))
        vis = float(getattr(lm, "visibility", 1.0))
        vis_list.append(vis)
        lms.append({"x": ox, "y": oy, "visibility": vis})
    return lms, float(np.mean(vis_list))

## 3. Carga del Dataset + Experimento de Ablación PDI (Sin vs. Con CLAHE + Letterboxing)

In [ ]:
def match_class(folder_name: str) -> Optional[str]:
    low = folder_name.lower().strip()
    for cls, aliases in CLASS_ALIASES.items():
        if any(a in low for a in aliases):
            return cls
    return None

def collect_images(root: Path) -> Dict[str, List[Tuple[Path, int]]]:
    splits = {"train": [], "val": [], "test": []}
    has_split_dirs = all((root / s).exists() for s in ("train", "val", "test"))
    if has_split_dirs:
        for s in ("train", "val", "test"):
            for cls_dir in (root / s).iterdir():
                if not cls_dir.is_dir():
                    continue
                cls = match_class(cls_dir.name)
                if cls is None:
                    continue
                label = SCULPTURE_CLASSES.index(cls)
                for p in sorted(cls_dir.rglob("*")):
                    if p.suffix.lower() in IMAGE_EXTS:
                        splits[s].append((p, label))
    else:
        # Split estratificado 80/10/10 si no hay subcarpetas train/val/test
        from sklearn.model_selection import train_test_split
        all_items = []
        for cls_dir in root.iterdir():
            if not cls_dir.is_dir():
                continue
            cls = match_class(cls_dir.name)
            if cls is None:
                continue
            label = SCULPTURE_CLASSES.index(cls)
            for p in sorted(cls_dir.rglob("*")):
                if p.suffix.lower() in IMAGE_EXTS:
                    all_items.append((p, label))
        labels = [y for _, y in all_items]
        tr, tmp = train_test_split(all_items, test_size=0.20, stratify=labels, random_state=SEED)
        va, te = train_test_split(tmp, test_size=0.50, stratify=[y for _, y in tmp], random_state=SEED)
        splits = {"train": tr, "val": va, "test": te}
    return splits

splits = collect_images(DATASET_ROOT)
for k, v in splits.items():
    print(f"{k}: {len(v)} imágenes")

# Extracción de features (con PDI) + ablación (sin PDI)
data = {}
ablation = {"raw_ok": 0, "pdi_ok": 0, "raw_vis": [], "pdi_vis": [], "total": 0}

for split_name, items in splits.items():
    X_list, y_list = [], []
    for img_path, label in items:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        ablation["total"] += 1

        raw_res = detect_pose(img, use_pdi=False)
        if raw_res is not None:
            ablation["raw_ok"] += 1
            ablation["raw_vis"].append(raw_res[1])

        pdi_res = detect_pose(img, use_pdi=True)
        if pdi_res is not None:
            ablation["pdi_ok"] += 1
            ablation["pdi_vis"].append(pdi_res[1])
            X_list.append(extract_feature_vector(pdi_res[0]))
            y_list.append(label)

    data[split_name] = (np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64))
    print(f"[{split_name}] detectadas con PDI: {len(X_list)}/{len(items)}")

print("\n=== ABLACIÓN PDI ===")
print(f"Sin PDI -> detección: {ablation['raw_ok']}/{ablation['total']} ({100*ablation['raw_ok']/max(1,ablation['total']):.1f}%) | vis media: {np.mean(ablation['raw_vis']):.3f}")
print(f"Con PDI -> detección: {ablation['pdi_ok']}/{ablation['total']} ({100*ablation['pdi_ok']/max(1,ablation['total']):.1f}%) | vis media: {np.mean(ablation['pdi_vis']):.3f}")

## 4. Entrenamiento de `PoseMLP` (con Data Augmentation en Espejo solo en `train`) y Comparativa de Modelos

In [ ]:
X_train_base, y_train_base = data["train"]
X_val, y_val = data["val"]
X_test, y_test = data["test"]

# Data augmentation por espejo horizontal ÚNICAMENTE sobre train (sin fuga hacia val/test)
X_train_mirror = np.array([mirror_feature_vector(v) for v in X_train_base], dtype=np.float32)
X_train = np.concatenate([X_train_base, X_train_mirror], axis=0)
y_train = np.concatenate([y_train_base, y_train_base], axis=0)
print("Train con espejo:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)

# 1) Random Forest y SVM (baselines de ML clásico sobre los 12 ángulos)
rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

svm = SVC(kernel="rbf", C=3.0, probability=True, random_state=SEED)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

# 2) PoseMLP (arquitectura EXACTA de backend/evaluator.py)
class PoseMLP(nn.Module):
    def __init__(self, in_features: int = 12, num_classes: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, num_classes),
        )
    def forward(self, x):
        return self.net(x)

model = PoseMLP(in_features=len(FEATURE_ORDER), num_classes=len(SCULPTURE_CLASSES)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

X_tr_t = torch.tensor(X_train, device=DEVICE)
y_tr_t = torch.tensor(y_train, device=DEVICE)
X_va_t = torch.tensor(X_val, device=DEVICE)
y_va_t = torch.tensor(y_val, device=DEVICE)
X_te_t = torch.tensor(X_test, device=DEVICE)

best_val_acc = -1.0
best_state = None
history = {"train_loss": [], "val_acc": []}

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    logits = model(X_tr_t)
    loss = criterion(logits, y_tr_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_va_t).argmax(dim=1).cpu().numpy()
        val_acc = accuracy_score(y_val, val_preds)
    history["train_loss"].append(float(loss.item()))
    history["val_acc"].append(float(val_acc))

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    mlp_pred = model(X_te_t).argmax(dim=1).cpu().numpy()

print("\n=== TABLA COMPARATIVA EN TEST ===")
print(f"CNN RGB (Parcial 2 baseline, val) -> Accuracy: 83.3%")
print(f"Random Forest (12 ángulos)        -> Accuracy: {100*accuracy_score(y_test, rf_pred):.1f}% | Macro-F1: {f1_score(y_test, rf_pred, average='macro'):.3f}")
print(f"SVM RBF (12 ángulos)              -> Accuracy: {100*accuracy_score(y_test, svm_pred):.1f}% | Macro-F1: {f1_score(y_test, svm_pred, average='macro'):.3f}")
print(f"PoseMLP PyTorch (12 ángulos)      -> Accuracy: {100*accuracy_score(y_test, mlp_pred):.1f}% | Macro-F1: {f1_score(y_test, mlp_pred, average='macro'):.3f}")
print("\nReporte detallado PoseMLP (Test):")
print(classification_report(y_test, mlp_pred, target_names=SCULPTURE_CLASSES))

## 5. Matriz de Confusión y Exportación de `pose_mlp.pt`

In [ ]:
cm = confusion_matrix(y_test, mlp_pred)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrBr", xticklabels=SCULPTURE_CLASSES, yticklabels=SCULPTURE_CLASSES, ax=axes[0])
axes[0].set_title("Matriz de Confusión · PoseMLP (Test)")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Clase Real")

axes[1].plot(history["train_loss"], label="Train Loss")
axes[1].plot(history["val_acc"], label="Val Accuracy")
axes[1].set_title("Curvas de Entrenamiento · PoseMLP")
axes[1].set_xlabel("Época")
axes[1].legend()
plt.tight_layout()
plt.show()

# Exportar checkpoint compatible con backend/evaluator.py
OUT_PT = Path("/content/pose_mlp.pt")
torch.save({
    "state_dict": best_state,
    "feature_order": FEATURE_ORDER,
    "classes": SCULPTURE_CLASSES,
    "test_accuracy": float(accuracy_score(y_test, mlp_pred)),
    "test_macro_f1": float(f1_score(y_test, mlp_pred, average="macro")),
}, OUT_PT)
print("Guardado en:", OUT_PT)

from google.colab import files
files.download(str(OUT_PT))